<a href="https://colab.research.google.com/github/nandininaik145-ini/Retail-Sales-Forcasting/blob/main/retailsales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# --- WEEK 1: PROJECT KICKOFF & DATA EXPLORATION ---

# 1. Load the datasets (Ensure these files are in the same folder as your code)
try:
    train = pd.read_csv('/content/train.csv', low_memory=False)
    store = pd.read_csv('/content/store.csv')
    print("✅ Files loaded successfully!")
except FileNotFoundError:
    print("❌ Error: Ensure 'train.csv' and 'store.csv' are in your directory.")

# 2. Merge the datasets (The most important step)
# This attaches store metadata to every single sales record
df = pd.merge(train, store, on='Store', how='left')

# 3. Quick Exploration
print(f"Total rows: {df.shape[0]}")
print(df.head())

# --- WEEK 2: DATA PREPROCESSING & CLEANING ---

# 1. Handle Missing Values (The 'Perfect' Way)
# Distance is usually better filled with Median to avoid outliers
df['CompetitionDistance'] = df['CompetitionDistance'].fillna(df['CompetitionDistance'].median())

# For these, 0 means there is no competition or promo active
df['CompetitionOpenSinceMonth'] = df['CompetitionOpenSinceMonth'].fillna(0)
df['CompetitionOpenSinceYear'] = df['CompetitionOpenSinceYear'].fillna(0)
df['Promo2SinceWeek'] = df['Promo2SinceWeek'].fillna(0)
df['Promo2SinceYear'] = df['Promo2SinceYear'].fillna(0)
df['PromoInterval'] = df['PromoInterval'].fillna("None")

# 2. Convert Date column and Extract Features
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)

# 3. Filter for Logic
# We only train on days the store was OPEN and had SALES > 0
df = df[df['Open'] != 0]
df = df[df['Sales'] > 0]

# 4. Handle Categorical Data (StateHoliday is often mixed type)
# Standardize '0' and 0 to be the same string
df['StateHoliday'] = df['StateHoliday'].astype(str).replace('0.0', '0')

print("\n✅ Data cleaning and feature engineering complete!")
print(df[['Date', 'Sales', 'Year', 'Month', 'CompetitionDistance']].head())

✅ Files loaded successfully!
Total rows: 1017209
   Store  DayOfWeek        Date  Sales  Customers  Open  Promo StateHoliday  \
0      1          5  2015-07-31   5263        555     1      1            0   
1      2          5  2015-07-31   6064        625     1      1            0   
2      3          5  2015-07-31   8314        821     1      1            0   
3      4          5  2015-07-31  13995       1498     1      1            0   
4      5          5  2015-07-31   4822        559     1      1            0   

   SchoolHoliday StoreType Assortment  CompetitionDistance  \
0              1         c          a               1270.0   
1              1         a          a                570.0   
2              1         a          a              14130.0   
3              1         c          c                620.0   
4              1         a          a              29910.0   

   CompetitionOpenSinceMonth  CompetitionOpenSinceYear  Promo2  \
0                        9.0         